# TCCON-like Retrieval Demo

Demonstrates a direct-sun (upward-looking) transmission retrieval using a
Bruker IFS125HR–style FTIR instrument configuration (`configs/tccon.yml`).

**Retrieval targets:** XCO₂, XCH₄, XH₂O — and XCO when the ABSCO table
is available (Phase 3a: `python prepare_absco.py --molecules co`).

**Structure**
1. Atmospheric profiles and ABSCO loading  
2. Build the TCCON instrument; filter windows to available ABSCO molecules  
3. Single-SZA demo (SZA = 30°): spectral fits, averaging kernels, DOF  
4. SZA sweep (10°–70°): retrieved column amounts, posterior uncertainties, DOF

**Requirements** (minimum, Phase 2/3b baseline):
```
python prepare_absco.py --molecules h2o_7755_8015,h2o_5995_6400,h2o_4750_5000,\
       h2o_4200_4350,co2_6100_6400,co2_4750_5000,ch4_4200_4350
```
For CO retrieval also run: `python prepare_absco.py --molecules n2o,co`

## 1  Setup

In [ ]:
import sys
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sys.path.insert(0, os.path.abspath('.'))

from gert.absco import ABSCOTable
from gert.atmosphere import AtmosphericProfile
from gert.forward_model import ForwardModel
from gert.geometry import Geometry
from gert.instrument import Instrument, SpectralWindow
from gert.instrument_config import InstrumentConfig
from gert.retrieval import GERTRetrieval, StateVector
from gert.rt_solver import TransmissionSolver

plt.rcParams.update({'font.size': 10, 'figure.dpi': 110})
print('Imports OK')

In [ ]:
# ── 11-level pressure grid, TOA → surface [Pa] ────────────────────────────────
p_lev = np.array([
    0.01, 10000., 20000., 30000., 40000., 50000.,
    60000., 70000., 80000., 90000., 100000.
])
n_lev = len(p_lev)

# Temperature [K]
T_lev = np.array([205.59104919, 203.52311895, 220.08876161, 242.37260452,
                   255.89316285, 266.14411523, 271.82797304, 277.33967816,
                   279.68627102, 283.79536257, 291.02379557])

# Specific humidity [kg/kg]
q_lev = np.array([3.17349759e-06, 3.60197244e-06, 5.41395304e-05,
                   6.18924547e-04, 1.51411285e-03, 8.10448167e-04,
                   4.89113692e-04, 1.82153489e-03, 7.49874173e-03,
                   8.96673750e-03, 1.03688360e-02])

# H₂O VMR from specific humidity [mol/mol]
_M_h2o, _M_air = 0.018015, 0.0289644
h2o_lev = q_lev * _M_air / ((1.0 - q_lev) * _M_h2o + q_lev * _M_air)

# CO₂ [mol/mol] — realistic mid-latitude profile
co2_lev = np.array([0.0004032, 0.00041564, 0.00041696, 0.00041686,
                     0.00041675, 0.00041665, 0.00041653, 0.00041641,
                     0.00041627, 0.00041612, 0.00041595])

# CH₄ [mol/mol] — tropospheric ~1900 ppb, stratospheric depletion
ch4_lev = np.array([500e-9, 1500e-9, 1900e-9, 1900e-9, 1900e-9,
                     1900e-9, 1900e-9, 1900e-9, 1900e-9, 1900e-9, 1900e-9])

# N₂O climatology [mol/mol]: ~330 ppb surface, ~100 ppb in stratosphere
n2o_lev = np.array([100e-9, 250e-9, 290e-9, 315e-9, 320e-9,
                     322e-9, 322e-9, 325e-9, 327e-9, 329e-9, 330e-9])

# CO climatology [mol/mol]: ~100 ppb surface, ~50 ppb in upper troposphere
co_lev = np.interp(p_lev, [p_lev[0], p_lev[-1]], [50e-9, 100e-9])

# O₂ mixing ratio for 1.27 µm A-band [mol/mol]
o2_1p27_lev = np.full(n_lev, 0.2095)

# ── Prior profiles (slightly perturbed from truth) ────────────────────────────
co2_prior_lev     = co2_lev * 0.99   # 1% low CO₂ prior
ch4_prior_lev     = ch4_lev * 1.05   # 5% high CH₄ prior
h2o_prior_lev     = h2o_lev * 1.00   # H₂O prior = truth
n2o_prior_lev     = n2o_lev * 1.00   # N₂O not retrieved; prior = truth
co_prior_lev      = co_lev  * 1.00   # CO prior = truth (large σ_prior)
o2_1p27_prior_lev = o2_1p27_lev

# Master dictionaries of true/prior gas profiles
TRUE_GASES_ALL = {
    'co2':     co2_lev,
    'ch4':     ch4_lev,
    'h2o':     h2o_lev,
    'n2o':     n2o_lev,
    'co':      co_lev,
    'o2_1p27': o2_1p27_lev,
}
PRIOR_GASES_ALL = {
    'co2':     co2_prior_lev,
    'ch4':     ch4_prior_lev,
    'h2o':     h2o_prior_lev,
    'n2o':     n2o_prior_lev,
    'co':      co_prior_lev,
    'o2_1p27': o2_1p27_prior_lev,
}
print(f'Atmospheric profile: {n_lev} levels,  '
      f'p = [{p_lev[0]:.2f}, {p_lev[-1]:.0f}] Pa')

## 2  ABSCO Tables and TCCON Instrument

In [ ]:
ABSCO_PATH = 'absco.h5'
tables = ABSCOTable.load_all(ABSCO_PATH, require_all=False)
print('Loaded ABSCO tables:', sorted(tables.keys()))
print()
print(f'  {"molecule":<12}  {"wn_min":>10}  {"wn_max":>10}  {"n_wn":>8}')
print('  ' + '-' * 46)
for mol, tab in sorted(tables.items()):
    print(f'  {mol:<12}  {tab.wavenumber[0]:>10.3f}  '
          f'{tab.wavenumber[-1]:>10.3f}  {len(tab.wavenumber):>8d}')

In [ ]:
SNR = 600.0
config          = InstrumentConfig.from_yaml('input/instruments/tccon.yml')
instrument_full = config.build_instrument(snr=SNR)

MIN_WINDOW_WIDTH = 20.0   # cm⁻¹ — windows trimmed to less than this are skipped

def _valid_range(tables, win, min_width=MIN_WINDOW_WIDTH):
    """
    Find the wavenumber range jointly covered by all molecule tables for a window.

    Boundary values are snapped to exact ABSCO table grid points via
    searchsorted so that downstream wn_index() exact-match lookups always
    succeed.  Returns (wn_lo, wn_hi) or None if coverage < min_width cm⁻¹.
    """
    wn_lo, wn_hi = win.wn_min, win.wn_max
    for mol in win.molecules:
        if mol not in tables:
            return None
        wn_arr = tables[mol].wavenumber
        i_lo = int(np.searchsorted(wn_arr, wn_lo, side='left'))
        if i_lo >= len(wn_arr):
            return None
        i_hi = int(np.searchsorted(wn_arr, wn_hi, side='right')) - 1
        if i_hi < 0 or i_hi < i_lo:
            return None
        wn_lo = float(wn_arr[i_lo])
        wn_hi = float(wn_arr[i_hi])
    if wn_hi - wn_lo < min_width:
        return None
    return wn_lo, wn_hi


print(f'TCCON instrument: {instrument_full.n_windows} windows defined in config\n')

filt_win_list = []
for win in instrument_full.windows:
    missing_mol = [m for m in win.molecules if m not in tables]
    if missing_mol:
        note   = f'  ← no ABSCO for {missing_mol}'
        status = 'SKIP'
        use_win = None
    else:
        valid = _valid_range(tables, win)
        if valid is None:
            note   = '  ← insufficient table coverage'
            status = 'SKIP'
            use_win = None
        else:
            wn_lo, wn_hi = valid
            use_win = SpectralWindow(
                wn_min=wn_lo, wn_max=wn_hi,
                ils=win.ils, molecules=win.molecules,
                label=win.label, channel_spacing_wn=win.channel_spacing_wn,
            )
            large_trim = wn_lo > win.wn_min + 0.5 or wn_hi < win.wn_max - 0.5
            if large_trim:
                note = (f'  [trimmed {win.wn_min:.0f}–{win.wn_max:.0f}'
                        f' → {wn_lo:.2f}–{wn_hi:.2f} cm⁻¹]')
            elif abs(wn_lo - win.wn_min) > 1e-6 or abs(wn_hi - win.wn_max) > 1e-6:
                note = f'  [snapped to table grid: {wn_lo:.4f}–{wn_hi:.4f} cm⁻¹]'
            else:
                note = ''
            status = 'USE '

    print(f'  [{status}]  {win.label:<12s}  '
          f'{win.wn_min:.0f}–{win.wn_max:.0f} cm⁻¹  '
          f'mols={win.molecules}{note}')
    if use_win is not None:
        filt_win_list.append(use_win)

if not filt_win_list:
    raise RuntimeError(
        'No windows available!  Run:\n'
        '  python prepare_absco.py --molecules '
        'o2_1p27,h2o_7755_8015,h2o_5995_6400,h2o_4750_5000,h2o_4200_4350,'
        'co2_6100_6400,co2_4750_5000,ch4_4200_4350'
    )

filt_instrument = Instrument(windows=filt_win_list, snr=SNR)
print(f'\n→ {filt_instrument.n_windows} windows active, '
      f'{filt_instrument.n_channels_total} total channels')

# Build AtmosphericProfile with only the gases referenced by active windows
all_needed_mols = set()
for win in filt_win_list:
    all_needed_mols.update(win.molecules)

def _build_atm(gases_dict, needed_mols):
    return AtmosphericProfile(
        p_levels=p_lev, T_levels=T_lev, q_levels=q_lev,
        gases={m: gases_dict[m] for m in needed_mols if m in gases_dict},
    )

true_atm  = _build_atm(TRUE_GASES_ALL,  all_needed_mols)
prior_atm = _build_atm(PRIOR_GASES_ALL, all_needed_mols)

# Gases to retrieve.  o2_1p27 acts as a column airmass proxy: it uses a
# fixed VMR (0.20935), so its scale factor directly maps to surface pressure
# (scale ≈ p_surface_true / p_prior).  Prior uncertainty 5% is generous;
# tightening it constrains XCO₂ / XCH₄ via the shared p-T Jacobians.
RETRIEVE_CANDIDATES = ['co2', 'ch4', 'h2o', 'co', 'o2_1p27']
retrieve_gases = [
    g for g in RETRIEVE_CANDIDATES
    if g in tables and any(g in w.molecules for w in filt_win_list)
]
GAS_UNCERTS = {'co2': 0.10, 'ch4': 0.20, 'h2o': 0.30, 'co': 0.50, 'o2_1p27': 0.05}
print(f'\nGases to retrieve: {retrieve_gases}')

_GAS_FAC   = {'co2': 1e6, 'o2_1p27': 100.0}   # others default to 1e9
_GAS_UNITS = {'co2': 'ppm', 'o2_1p27': '%'}    # others default to 'ppb'

print('\nTrue column-average VMRs:')
for mol in retrieve_gases:
    xval  = true_atm.column_xgas(mol)
    fac   = _GAS_FAC.get(mol, 1e9)
    units = _GAS_UNITS.get(mol, 'ppb')
    label = 'O2_1p27' if mol == 'o2_1p27' else mol.upper()
    print(f'  X{label:<8s} = {xval * fac:.4f} {units}')

## 3  Single-SZA Demo (SZA = 30°)

Simulate a noisy TCCON spectrum at SZA = 30°, retrieve gas column scale
factors, and inspect the spectral fit quality, averaging kernels, and
degrees of freedom.

In [ ]:
def run_tccon_retrieval(
    sza, true_atm, prior_atm, instrument, tables,
    retrieve_gases, gas_uncerts,
    noise_seed=None, verbose=False,
):
    """
    Simulate a noisy TCCON spectrum at a given SZA and retrieve it.

    Returns
    -------
    result  : RetrievalResult
    y_true  : ndarray — noiseless simulated spectrum
    y_obs   : ndarray — noisy observed spectrum
    fr_true : ForwardResult — true forward model result (R_band, wl_band)
    """
    n_wins = instrument.n_windows
    geo    = Geometry(sza=sza, vza=0.0, raa=0.0)
    zeros  = np.zeros(n_wins)

    fr_true = ForwardModel(
        atm=true_atm, absco_tables=tables,
        instrument=instrument, geometry=geo,
        solver=TransmissionSolver(),
    ).run(albedo=zeros)
    y_true = fr_true.y.copy()

    Sy          = instrument.build_Sy(fr_true.R_band)
    sigma_noise = np.sqrt(np.diag(Sy))
    seed  = noise_seed if noise_seed is not None else int(sza * 1000 + 1)
    y_obs = y_true + np.random.default_rng(seed).standard_normal(len(y_true)) * sigma_noise
    Sy_inv = np.diag(1.0 / np.diag(Sy))

    sv = StateVector.transmission_scaling(
        gases=retrieve_gases,
        gas_uncerts={g: gas_uncerts.get(g, 0.10) for g in retrieve_gases},
        T_uncert=5.0,
        p_uncert=0.10,
    )

    fm_prior = ForwardModel(
        atm=prior_atm, absco_tables=tables,
        instrument=instrument, geometry=geo,
        solver=TransmissionSolver(),
    )

    result = GERTRetrieval(
        fm_prior=fm_prior,
        y_true=y_obs,
        Sy_inv=Sy_inv,
        state_vector=sv,
        prior_albedo=zeros,
        verbose=verbose,
        max_iter=15,
    ).run()

    return result, y_true, y_obs, fr_true


def split_y(y_vec, wins):
    """Split the concatenated measurement vector into per-window arrays."""
    bands, offset = [], 0
    for win in wins:
        bands.append(y_vec[offset:offset + win.n_channels])
        offset += win.n_channels
    return bands


def extract_xgas(result, prior_atm, mol):
    """
    Return (x_ret, sigma_post) in display units for a retrieved gas.

    Units: ppm for CO₂; % for o2_1p27; ppb for everything else.
    """
    names = result.state_vector.names
    key   = f'{mol}_scale'
    if key not in names:
        return None, None
    idx    = names.index(key)
    scale  = result.x_ret[idx]
    sigma  = result.posterior_sigma()[idx]
    xprior = prior_atm.column_xgas(mol)
    fac    = _GAS_FAC.get(mol, 1e9)
    return xprior * scale * fac, xprior * sigma * fac


print('Helper functions defined')

In [ ]:
SZA_DEMO = 30.0
print(f'Running TCCON retrieval at SZA = {SZA_DEMO}°  (verbose=True)\n')

result_demo, y_true_demo, y_obs_demo, fr_demo = run_tccon_retrieval(
    sza=SZA_DEMO,
    true_atm=true_atm, prior_atm=prior_atm,
    instrument=filt_instrument, tables=tables,
    retrieve_gases=retrieve_gases, gas_uncerts=GAS_UNCERTS,
    noise_seed=42, verbose=True,
)
print()
print(result_demo.summary())

print('\n── Column-average VMR comparison ─────────────────────────────')
print(f'{"Gas":<10}  {"True":>10}  {"Prior":>10}  {"Retrieved":>10}  {"±σ_post":>10}  units')
print('-' * 72)
for mol in retrieve_gases:
    xval_true  = true_atm.column_xgas(mol)
    xval_prior = prior_atm.column_xgas(mol)
    xval_ret, sigma_ret = extract_xgas(result_demo, prior_atm, mol)
    fac   = _GAS_FAC.get(mol, 1e9)
    units = _GAS_UNITS.get(mol, 'ppb')
    label = 'O2_1p27' if mol == 'o2_1p27' else mol.upper()
    if xval_ret is not None:
        print(f'{label:<10s}  {xval_true*fac:>10.4f}  {xval_prior*fac:>10.4f}  '
              f'{xval_ret:>10.4f}  {sigma_ret:>10.5f}  {units}')

In [ ]:
# ── Spectral fit plots ────────────────────────────────────────────────────────
n_wins = filt_instrument.n_windows

y_true_bands = split_y(y_true_demo,         filt_win_list)
y_obs_bands  = split_y(y_obs_demo,          filt_win_list)
y_ret_bands  = split_y(result_demo.y_ret,   filt_win_list)

fig, axes = plt.subplots(2, n_wins, figsize=(5.5 * n_wins, 6), squeeze=False)

for i, win in enumerate(filt_win_list):
    wl = fr_demo.wl_band[i]   # µm, blue→red

    # Upper panel: spectra
    ax = axes[0, i]
    ax.plot(wl, y_true_bands[i], 'k-',   lw=1.0, zorder=3, label='True')
    ax.plot(wl, y_obs_bands[i],  '.',    color='0.65', ms=1.2, alpha=0.7,
            zorder=2, label='Observed')
    ax.plot(wl, y_ret_bands[i],  'C1--', lw=1.3, zorder=4, label='Retrieved')
    ax.set_title(f'{win.label}\n{win.wn_min:.0f}–{win.wn_max:.0f} cm⁻¹',
                 fontsize=9)
    ax.set_ylabel('Attenuated irradiance\n[W m⁻² µm⁻¹]')
    ax.set_xlim(wl[0], wl[-1])
    ax.tick_params(labelbottom=False)
    if i == 0:
        ax.legend(loc='lower right', fontsize=8, markerscale=3)

    # Lower panel: normalised residual
    ax2 = axes[1, i]
    resid = y_obs_bands[i] - y_ret_bands[i]
    rms   = float(np.std(resid))
    ax2.plot(wl, resid / rms, '-', color='C0', lw=0.7)
    ax2.axhline(0,  color='k',    lw=0.9, ls='--')
    ax2.axhline(+1, color='gray', lw=0.6, ls=':')
    ax2.axhline(-1, color='gray', lw=0.6, ls=':')
    ax2.set_ylim(-4, 4)
    ax2.set_xlim(wl[0], wl[-1])
    ax2.set_xlabel('Wavelength [µm]')
    ax2.set_ylabel('Residual / RMS')

fig.suptitle(
    f'TCCON spectral fits — SZA = {SZA_DEMO}°   '
    f'SNR = {SNR:.0f}   '
    f'χ²_red = {result_demo.chisq_reduced:.3f}',
    fontsize=12, fontweight='bold',
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Averaging kernels and degrees of freedom ──────────────────────────────────
names  = result_demo.state_vector.names
dof_v  = result_demo.dof()        # diagonal of A
sig_pr = np.sqrt(np.diag(result_demo.Sa))
sig_po = result_demo.posterior_sigma()

# Label elements neatly
LABELS = {
    'co2_scale':  'CO₂ scale',
    'ch4_scale':  'CH₄ scale',
    'h2o_scale':  'H₂O scale',
    'co_scale':   'CO scale',
    'n2o_scale':  'N₂O scale',
    'T_offset':   'T offset',
    'p_scale':    'p scale',
}
labels = [LABELS.get(n, n) for n in names]
n_elem = len(names)
x_pos  = np.arange(n_elem)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: DOF bar chart
ax = axes[0]
colors = ['C0' if 'scale' in n and n != 'p_scale' else 'C3' for n in names]
ax.bar(x_pos, dof_v, color=colors, edgecolor='k', linewidth=0.6)
ax.set_xticks(x_pos)
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Degrees of freedom for signal (A_ii)')
ax.set_title(f'DOF per element   (Total = {dof_v.sum():.2f})')
ax.axhline(1.0, color='k', ls='--', lw=0.8, label='AK = 1 (full information)')
ax.set_ylim(0, 1.1)
ax.legend(fontsize=8)

# Right: prior vs posterior uncertainty comparison
ax2 = axes[1]
width = 0.35
ax2.bar(x_pos - width/2, sig_pr, width, color='C7', edgecolor='k',
        linewidth=0.5, label='Prior σ')
ax2.bar(x_pos + width/2, sig_po, width, color='C0', edgecolor='k',
        linewidth=0.5, label='Posterior σ')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
ax2.set_ylabel('1-sigma uncertainty (state-vector units)')
ax2.set_title('Prior vs posterior uncertainty')
ax2.legend()
ax2.set_yscale('log')

fig.suptitle(f'Averaging kernels and uncertainty — SZA = {SZA_DEMO}°',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4  Effect of SZA on Absorption Depth

As the solar zenith angle increases, the one-way solar air mass
`m = 1/cos(SZA)` increases, deepening absorption lines.  The plot below
shows this for the CO₂ 1.61 µm window (always available).

In [ ]:
SZA_SHOW = [10.0, 30.0, 50.0, 70.0]
colors_sza = ['C0', 'C1', 'C2', 'C3']

# Find the CO₂ 1.61 µm window (or fall back to first available)
demo_win_idx = 0
for k, win in enumerate(filt_win_list):
    if 'CO2_1p61' in win.label or 'co2' in win.molecules:
        demo_win_idx = k
        break

demo_win = filt_win_list[demo_win_idx]
fig, ax  = plt.subplots(figsize=(9, 4))

for sza_val, col in zip(SZA_SHOW, colors_sza):
    fr_sza = ForwardModel(
        atm=true_atm, absco_tables=tables,
        instrument=filt_instrument, geometry=Geometry(sza=sza_val, vza=0.0),
        solver=TransmissionSolver(),
    ).run(albedo=np.zeros(filt_instrument.n_windows))
    wl_b = fr_sza.wl_band[demo_win_idx]
    R_b  = fr_sza.R_band[demo_win_idx]
    ax.plot(wl_b, R_b, color=col, lw=1.2, label=f'SZA = {sza_val:.0f}°')

ax.set_xlabel('Wavelength [µm]')
ax.set_ylabel('Attenuated irradiance [W m⁻² µm⁻¹]')
ax.set_title(f'{demo_win.label} window — absorption depth vs SZA\n'
             f'({demo_win.wn_min:.0f}–{demo_win.wn_max:.0f} cm⁻¹,  '
             f'mols = {demo_win.molecules})')
ax.legend()
plt.tight_layout()
plt.show()

## 5  SZA Sweep (10°–70°)

Run the retrieval at each SZA in `SZA_GRID` and record:
- Retrieved column-average VMRs (XCO₂, XCH₄, XH₂O, XCO)
- Posterior 1-sigma uncertainties
- Degrees of freedom for signal
- Convergence status and χ²_red

In [ ]:
SZA_GRID = np.arange(10.0, 75.0, 30.0)   # 10, 20, 30, 40, 50, 60, 70

# Storage: keyed by molecule
xgas_ret   = {mol: [] for mol in retrieve_gases}
xgas_sigma = {mol: [] for mol in retrieve_gases}
dof_total  = []
dof_per_el = []   # list of dof vectors (one per SZA)
chisq_list = []
converged_list = []

print(f'SZA sweep: {SZA_GRID}  ({len(SZA_GRID)} retrievals)\n')
print(f'{"SZA":>6}  {"conv":>5}  {"χ²_red":>7}  '
      + '  '.join(f"{mol.upper():>8}" for mol in retrieve_gases))
print('-' * (30 + 10 * len(retrieve_gases)))

for sza_val in SZA_GRID:
    res, _, _, _ = run_tccon_retrieval(
        sza=sza_val,
        true_atm=true_atm,
        prior_atm=prior_atm,
        instrument=filt_instrument,
        tables=tables,
        retrieve_gases=retrieve_gases,
        gas_uncerts=GAS_UNCERTS,
        noise_seed=int(sza_val * 100),
        verbose=False,
    )
    dof_v   = res.dof()
    dof_total.append(dof_v.sum())
    dof_per_el.append(dof_v)
    chisq_list.append(res.chisq_reduced)
    converged_list.append(res.converged)

    gas_strs = []
    for mol in retrieve_gases:
        xr, xs = extract_xgas(res, prior_atm, mol)
        xgas_ret[mol].append(xr)
        xgas_sigma[mol].append(xs)
        gas_strs.append(f'{xr:8.2f}' if xr is not None else '        ')

    conv_str = 'YES' if res.converged else 'NO '
    print(f'{sza_val:6.1f}  {conv_str:>5}  {res.chisq_reduced:7.4f}  '
          + '  '.join(gas_strs))

# Convert to arrays
SZA_GRID_arr = np.asarray(SZA_GRID)
for mol in retrieve_gases:
    xgas_ret[mol]   = np.array(xgas_ret[mol],   dtype=float)
    xgas_sigma[mol] = np.array(xgas_sigma[mol], dtype=float)
dof_total = np.array(dof_total)
chisq_arr = np.array(chisq_list)

In [ ]:
# ── Plot retrieved column VMRs vs SZA ─────────────────────────────────────────
# Build GAS_META only for gases that are actually in the active atmosphere,
# so calling column_xgas() never raises KeyError for inactive windows.
_GAS_META_DEFS = {
    'co2':     {'label': 'XCO₂ [ppm]',  'color': 'C0', 'fac': 1e6},
    'ch4':     {'label': 'XCH₄ [ppb]',  'color': 'C2', 'fac': 1e9},
    'h2o':     {'label': 'XH₂O [ppb]',  'color': 'C1', 'fac': 1e9},
    'co':      {'label': 'XCO [ppb]',   'color': 'C3', 'fac': 1e9},
    'o2_1p27': {'label': 'XO₂ [%]',     'color': 'C4', 'fac': 100.0},
}
GAS_META = {}
for mol in retrieve_gases:
    if mol in _GAS_META_DEFS and mol in true_atm.gases:
        d = _GAS_META_DEFS[mol]
        GAS_META[mol] = {**d, 'true': true_atm.column_xgas(mol) * d['fac']}

active_gases = list(GAS_META.keys())
ncols = min(len(active_gases), 2)
nrows = (len(active_gases) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)

for idx, mol in enumerate(active_gases):
    meta = GAS_META[mol]
    ax   = axes[idx // ncols, idx % ncols]
    xr   = xgas_ret[mol]
    xs   = xgas_sigma[mol]

    ax.errorbar(SZA_GRID_arr, xr, yerr=xs, fmt='o-', color=meta['color'],
                capsize=4, capthick=1.2, lw=1.5, ms=5, label='Retrieved ±1σ')
    ax.axhline(meta['true'], color='k', lw=1.2, ls='--', label='True')
    xprior = prior_atm.column_xgas(mol) * meta['fac']
    ax.axhline(xprior, color='0.5', lw=1.0, ls=':', label='Prior')

    ax.set_xlabel('Solar zenith angle [°]')
    ax.set_ylabel(meta['label'])
    ax.set_title(meta['label'].split(' ')[0])
    ax.set_xticks(SZA_GRID_arr)
    ax.legend(fontsize=8)

for k in range(len(active_gases), nrows * ncols):
    axes[k // ncols, k % ncols].set_visible(False)

fig.suptitle(f'Retrieved column VMRs vs SZA   (SNR = {SNR:.0f})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Posterior 1-sigma vs SZA ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))

for mol in active_gases:
    meta = GAS_META[mol]
    xs   = xgas_sigma[mol]
    ax.plot(SZA_GRID_arr, xs, 'o-', color=meta['color'], lw=1.5, ms=5,
            label=meta['label'].split(' ')[0])

ax.set_xlabel('Solar zenith angle [°]')
ax.set_ylabel('Posterior 1-sigma (retrieval units)')
ax.set_title(f'Posterior uncertainties vs SZA   (SNR = {SNR:.0f})\n'
             'Larger SZA → more path → smaller σ')
ax.set_xticks(SZA_GRID_arr)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── DOF vs SZA ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Total DOF
ax = axes[0]
ax.plot(SZA_GRID_arr, dof_total, 's-', color='k', lw=1.5, ms=6)
ax.set_xlabel('Solar zenith angle [°]')
ax.set_ylabel('Total DOF for signal')
ax.set_title('Total degrees of freedom vs SZA')
ax.set_xticks(SZA_GRID_arr)
ax.set_ylim(0, None)
ax2t = ax.twinx()
ax2t.plot(SZA_GRID_arr, chisq_arr, 'd--', color='C3', lw=1.0, ms=5)
ax2t.set_ylabel('χ²_red', color='C3')
ax2t.tick_params(axis='y', colors='C3')
ax2t.axhline(1.0, color='C3', ls=':', lw=0.8)

# Per-element DOF stacked (gas elements only)
ax2 = axes[1]
dof_arr = np.array(dof_per_el)   # shape (n_sza, n_elem)
sv_names  = result_demo.state_vector.names   # from the demo run
gas_idxs  = [i for i, n in enumerate(sv_names) if n.endswith('_scale') and n != 'p_scale']
gas_names = [sv_names[i] for i in gas_idxs]
gas_label = [LABELS.get(n, n) for n in gas_names]
gas_colors_map = {
    'co2_scale': 'C0', 'ch4_scale': 'C2', 'h2o_scale': 'C1',
    'co_scale': 'C3', 'p_scale': 'C4',
}
bottom = np.zeros(len(SZA_GRID_arr))
for ii, (gi, gl) in enumerate(zip(gas_idxs, gas_label)):
    col = gas_colors_map.get(sv_names[gi], f'C{ii}')
    ax2.bar(SZA_GRID_arr, dof_arr[:, gi], bottom=bottom,
            width=6.0, color=col, edgecolor='k', linewidth=0.5,
            label=gl, alpha=0.85)
    bottom += dof_arr[:, gi]

ax2.set_xlabel('Solar zenith angle [°]')
ax2.set_ylabel('DOF for signal')
ax2.set_title('Per-gas DOF vs SZA')
ax2.set_xticks(SZA_GRID_arr)
ax2.set_ylim(0, None)
ax2.legend(fontsize=8, loc='upper left')

fig.suptitle('Degrees of freedom for signal vs SZA',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── O₂ 1.27 µm as surface-pressure proxy ─────────────────────────────────────
# XO₂ uses a fixed VMR (0.20935), so the retrieved scale factor equals
#   scale = τ_retrieved / τ_prior  ≈  p_surface_true / p_surface_prior
# Deviations from 1.0 indicate systematic errors: surface pressure bias,
# spectroscopy error, ILS mis-characterisation, or unmodelled aerosol.
if 'o2_1p27' not in retrieve_gases:
    print('O₂ 1.27 µm not retrieved — run prepare_absco.py --molecules o2_1p27,h2o_7755_8015')
else:
    o2_prior_fac  = prior_atm.column_xgas('o2_1p27') * 100.0   # in %
    o2_scale      = xgas_ret['o2_1p27']   / o2_prior_fac        # dimensionless scale
    o2_scale_sig  = xgas_sigma['o2_1p27'] / o2_prior_fac

    p_true     = float(p_lev[-1])           # true surface pressure [Pa]
    p_inferred = p_true * o2_scale          # inferred p_surface [Pa]
    p_sigma    = p_true * o2_scale_sig      # posterior 1-sigma [Pa]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Left: O₂ scale factor vs SZA
    ax = axes[0]
    ax.errorbar(SZA_GRID_arr, o2_scale, yerr=o2_scale_sig, fmt='o-', color='C4',
                capsize=4, capthick=1.2, lw=1.5, ms=5, label='Retrieved ±1σ')
    ax.axhline(1.0, color='k', lw=1.2, ls='--', label='True (scale = 1)')
    ax.set_xlabel('Solar zenith angle [°]')
    ax.set_ylabel('O₂ column scale factor')
    ax.set_title('O₂ 1.27 µm scale factor\n'
                 '(proxy for column airmass / surface pressure)')
    ax.set_xticks(SZA_GRID_arr)
    ax.legend(fontsize=9)

    # Right: inferred surface pressure vs SZA
    ax2 = axes[1]
    ax2.errorbar(SZA_GRID_arr, p_inferred / 100, yerr=p_sigma / 100, fmt='s-', color='C4',
                 capsize=4, capthick=1.2, lw=1.5, ms=5, label='XO₂-inferred p_surf ±1σ')
    ax2.axhline(p_true / 100, color='k', lw=1.2, ls='--',
                label=f'True surface pressure ({p_true/100:.1f} hPa)')
    ax2.set_xlabel('Solar zenith angle [°]')
    ax2.set_ylabel('Surface pressure [hPa]')
    ax2.set_title('Inferred surface pressure from XO₂\n'
                  'Bias → systematic retrieval error')
    ax2.set_xticks(SZA_GRID_arr)
    ax2.legend(fontsize=9)

    fig.suptitle('O₂ 1.27 µm as airmass / pressure proxy   '
                 f'(SNR = {SNR:.0f})',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f'\n── XO₂ surface-pressure proxy ─────────────────────────────')
    print(f'True surface pressure: {p_true:.1f} Pa  ({p_true/100:.2f} hPa)')
    print(f'\n{"SZA":>6}  {"O₂ scale":>10}  {"±σ_scale":>10}  '
          f'{"p_infer [hPa]":>14}  {"±σ [hPa]":>10}')
    print('-' * 58)
    for sza_v, sc, ss, pi, si in zip(
            SZA_GRID_arr, o2_scale, o2_scale_sig, p_inferred, p_sigma):
        print(f'{sza_v:6.1f}  {sc:10.5f}  {ss:10.5f}  '
              f'{pi/100:14.3f}  {si/100:10.4f}')

In [ ]:
# ── Retrieval bias vs SZA ────────────────────────────────────────────────────
# Bias = retrieved - true, in units of posterior sigma
fig, ax = plt.subplots(figsize=(7, 4))
ax.axhline(0,  color='k',    lw=1.0, ls='-')
ax.axhline(+1, color='gray', lw=0.8, ls='--')
ax.axhline(-1, color='gray', lw=0.8, ls='--', label='±1σ')

for mol in active_gases:
    meta   = GAS_META[mol]
    xr     = xgas_ret[mol]
    xs     = xgas_sigma[mol]
    xtrue  = meta['true']
    bias   = (xr - xtrue) / xs   # in units of posterior sigma
    ax.plot(SZA_GRID_arr, bias, 'o-', color=meta['color'], lw=1.3, ms=5,
            label=meta['label'].split(' ')[0])

ax.set_xlabel('Solar zenith angle [°]')
ax.set_ylabel('(Retrieved − True) / σ_post')
ax.set_title('Retrieval bias (noise-realisation dependent)\n'
             'Values within ±1 indicate consistency with noise')
ax.set_xticks(SZA_GRID_arr)
ax.set_ylim(-3.5, 3.5)
ax.legend()
plt.tight_layout()
plt.show()
print('Done.')